# Pipeline de Minería de Datos – Minería de Datos II / IC-2026
**CUC – Colegio Universitario de Cartago**  
Curso: Big Data (BD-162) | Grupo #2

Este notebook ejecuta el pipeline completo sobre **cualquier archivo CSV**:
1. Carga de datos
2. Limpieza de datos
3. Transformación de datos
4. Análisis exploratorio (EDA)
5. Modelo Apriori
6. Modelo ECLAT
7. Sistema de recomendación y propuestas de negocio

## 0. Instalación de dependencias

In [ ]:
# Ejecutar una sola vez si no tiene las dependencias instaladas
# !pip install mlxtend networkx matplotlib seaborn pandas numpy

## 1. Configuración – Ingrese aquí su CSV y parámetros

In [ ]:
import sys, os
# Agregar raíz del proyecto al path para importar 'src'
sys.path.insert(0, os.path.abspath('..'))

from src.pipeline.mining_pipeline import MiningPipeline

# ─────────────────────────────────────────────────────────────────────
# ▶  CONFIGURE AQUÍ SU ARCHIVO Y PARÁMETROS
# ─────────────────────────────────────────────────────────────────────
CSV_PATH = '../data/raw/Software.csv'   # <-- Cambie por su archivo CSV

# Columnas que formarán las transacciones (None = inferencia automática)
RULE_COLUMNS = ['Region', 'Subregion', 'Industry', 'Segment', 'Product']
# RULE_COLUMNS = None  # Descomente esta línea para inferencia automática

# Parámetros de los algoritmos
MIN_SUPPORT_APRIORI    = 0.1   # 10% de soporte
MIN_CONFIDENCE_APRIORI = 0.5   # 50% de confianza
MIN_SUPPORT_ECLAT      = 0.2   # 20% de soporte
MIN_CONFIDENCE_ECLAT   = 0.5   # 50% de confianza

# Estrategia de nulos: 'drop' | 'fill' | 'none'
NULL_STRATEGY = 'drop'

# Guardar plots como imágenes en reports/
SAVE_PLOTS = True
# ─────────────────────────────────────────────────────────────────────

## 2. Ejecución del Pipeline Completo

In [ ]:
pipeline = MiningPipeline(
    csv_path               = CSV_PATH,
    rule_columns           = RULE_COLUMNS,
    min_support_apriori    = MIN_SUPPORT_APRIORI,
    min_confidence_apriori = MIN_CONFIDENCE_APRIORI,
    min_support_eclat      = MIN_SUPPORT_ECLAT,
    min_confidence_eclat   = MIN_CONFIDENCE_ECLAT,
    null_strategy          = NULL_STRATEGY,
    save_plots             = SAVE_PLOTS,
).run()

## 3. Resultados – Top Reglas Apriori

In [ ]:
print('Top 10 reglas Apriori (ordenadas por lift):')
pipeline.apriori_.get_top_rules(10)

## 4. Resultados – Top Reglas ECLAT

In [ ]:
print('Top 10 reglas ECLAT (ordenadas por lift):')
pipeline.eclat_.get_top_rules(10)

## 5. Sistema de Recomendación

In [ ]:
# Recomendación basada en ítems específicos (Apriori)
# Cambie los ítems por valores presentes en su CSV
items_consulta = ['Software', 'LATAM']

print(f'Recomendaciones para: {items_consulta}')
rec_apriori = pipeline.recommend(items_consulta, model='apriori')
rec_apriori

In [ ]:
# Misma consulta con ECLAT
rec_eclat = pipeline.recommend(items_consulta, model='eclat')
rec_eclat

## 6. Propuestas de Negocio

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 200)

print('=== TOP 10 PROPUESTAS DE NEGOCIO (Apriori) ===')
propuestas = pipeline.top_proposals(model='apriori', n=10)
for i, row in propuestas.iterrows():
    print(f'\n{i+1}. {row["propuesta"]}')

In [ ]:
print('=== TOP 10 PROPUESTAS DE NEGOCIO (ECLAT) ===')
propuestas_eclat = pipeline.top_proposals(model='eclat', n=10)
for i, row in propuestas_eclat.iterrows():
    print(f'\n{i+1}. {row["propuesta"]}')

## 7. Exportar reglas para el Dashboard

In [ ]:
from src.utils.config import RULES_TECH_FILE, RULES_RETAIL_FILE

# ─────────────────────────────────────────────────────────────────────
# Exportar al CSV que el dashboard consume
# Cambie la ruta según el dataset que procesó:
#   RULES_TECH_FILE   → Tecnología / Software
#   RULES_RETAIL_FILE → Retail (cadena detallista)
# ─────────────────────────────────────────────────────────────────────
pipeline.export_rules_for_dashboard(RULES_TECH_FILE)

# Para retail, repita el pipeline con su CSV de retail y exporte:
# pipeline_retail.export_rules_for_dashboard(RULES_RETAIL_FILE)
print("Listo. Ahora puede lanzar el dashboard con:")
print("  cd mining_pipeline && python dash_app/app.py")